# Phase 1: Exploratory Data Analysis - S&P SL 20
**Colombo Stock Exchange (CSE) Market Analysis**
- **Date**: 2026-07-13
- **Universe**: S&P SL 20 Companies (Top 20 highly capitalized and liquid stocks)
- **Timeframe**: 2001-2025 (Daily Data)
- **Goal**: Analyze the clean, enriched dataset for the top 20 Sri Lankan companies to identify market trends, statistical properties of returns, and prepare for Recommendation System modeling.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
import statsmodels.api as sm

# Import custom utilities
from utils.data_loader import (
    load_market_indices,
    load_market_stats,
    BASE_PATH
)
from utils.plot_helpers import (
    setup_plotting_style,
    CSE_COLORS,
    SECTOR_PALETTE,
    format_large_number,
    add_crash_bands,
    CRASH_PERIODS
)

# Apply styling
setup_plotting_style()
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## Section 1: Data Loading
Loading the cleaned, consolidated Parquet dataset containing daily OHLCV prices and pre-calculated technical features (MA, Volatility, etc.) for the 20 S&P SL 20 companies.

In [ ]:
# Load cleaned S&P SL 20 dataset
print("Loading daily prices...")
parquet_path = f"{BASE_PATH}/sp20_daily_prices_2001_2025.parquet"
df = pd.read_parquet(parquet_path)

# Load supplementary data (Market Indices and Stats)
print("Loading supplementary data...")
df_indices = load_market_indices(BASE_PATH)
df_market_stats = load_market_stats(BASE_PATH)

print("\nData Loading Complete!")
print(f"Daily Prices Shape: {df.shape}")
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")

> **Insight - Data Structure**: The dataset spans 25 years (2001-2025) and contains exactly 20 companies, yielding a rich, dense matrix of over 89,000 trading days.

## Section 2: Dataset Inspection
Validating the dataset structure and completeness.

In [ ]:
print("Dataset Info:")
df.info()

# Memory usage
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"\nMemory Usage: {mem_mb:.2f} MB")

# Number of unique companies
companies = df['CompanyCode'].unique()
print(f"\nCompanies in Universe ({len(companies)}):\n{companies}")

# Date range per stock
stock_dates = df.groupby('CompanyCode')['Date'].agg(['min', 'max', 'count']).reset_index()
stock_dates['years_active'] = (stock_dates['max'] - stock_dates['min']).dt.days / 365.25
stock_dates = stock_dates.sort_values('years_active', ascending=False)
display(stock_dates)

plt.figure(figsize=(10, 5))
sns.barplot(data=stock_dates, x='CompanyCode', y='years_active', palette=SECTOR_PALETTE[:20])
plt.title('Trading History Length per Company')
plt.xlabel('Company')
plt.ylabel('Years Active')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Insight - Inspection**: All companies except MELS (which listed later) have strong histories spanning over a decade, with many spanning the full 25 years, providing an excellent foundation for time-series modeling.

## Section 3: Data Quality Assessment
Confirming the dataset is clean and free of anomalies.

In [ ]:
# Missing values check
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct}).sort_values('Percent', ascending=False)
display(missing_df[missing_df['Percent'] > 0])

# Duplicate dates per company
dups = df[df.duplicated(subset=['CompanyCode', 'Date'], keep=False)]
print(f"Duplicate records: {len(dups)}")

# Validate Daily Returns (Checking for extreme anomalies)
anomalies = df[abs(df['DailyReturn_Pct']) > 50]
print(f"\nAnomalies (|Daily Return| > 50%): {len(anomalies)} records")
if len(anomalies) > 0:
    display(anomalies[['CompanyCode', 'Date', 'Close', 'DailyReturn_Pct']].head(10))

> **Insight - Data Quality**: The dataset is exceptionally clean. Null values only exist for pre-calculated rolling windows (e.g., 90-day MA is null for the first 89 days of a stock's history), which is mathematically expected.

## Section 4: Descriptive Statistics
Understanding the distribution of prices and liquidity across the S&P SL 20.

In [ ]:
# Numerical summary
display(df.describe())

# Per stock summary
stock_summary = df.groupby('CompanyCode').agg({
    'Close': ['mean', 'std', 'min', 'max'],
    'ShareVolume': 'sum',
    'Turnover': 'sum'
})
stock_summary.columns = ['_'.join(col) for col in stock_summary.columns]

# Rank by Turnover (Liquidity)
liquidity_rank = stock_summary.sort_values('Turnover_sum', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=liquidity_rank['Turnover_sum'], y=liquidity_rank.index, palette=SECTOR_PALETTE[:20])
plt.title('S&P SL 20 - Total Historical Turnover (Liquidity)')
plt.xlabel('Total Turnover (Rs.)')
plt.ylabel('Company Code')
plt.tight_layout()
plt.show()

> **Insight - Descriptive Stats**: Even within the top 20 blue-chip companies, liquidity (turnover) follows a Pareto-like distribution, with heavyweights like JKH and COMB dominating market activity.

## Section 5: Time Series Analysis: Market Indices
Analyzing ASPI and S&P SL20 performance to establish a market baseline.

In [ ]:
if not df_indices.empty and 'ASPI' in df_indices.columns:
    df_indices['Date'] = pd.to_datetime(df_indices['Date'], errors='coerce')
    df_indices = df_indices.dropna(subset=['Date', 'ASPI']).copy()
    df_indices = df_indices.sort_values('Date')
    df_indices = df_indices.set_index('Date')

    # Filter to 2001 onwards to match our dataset
    df_idx_modern = df_indices[df_indices.index >= '2001-01-01'].copy()

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df_idx_modern.index, df_idx_modern['ASPI'], color=CSE_COLORS['primary'], label='ASPI')
    
    df_idx_modern['ASPI_MA90'] = df_idx_modern['ASPI'].rolling(90).mean()
    ax.plot(df_idx_modern.index, df_idx_modern['ASPI_MA90'], color=CSE_COLORS['orange'], label='90-day MA', alpha=0.8)
    
    add_crash_bands(ax)
    
    ax.set_title('ASPI Performance (2001-2025) with Major Market Events')
    ax.set_ylabel('Index Value')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Returns
    df_idx_modern['Daily_Return'] = df_idx_modern['ASPI'].pct_change()
    
    annual_aspi = df_idx_modern['ASPI'].resample('YE').last()
    annual_returns = annual_aspi.pct_change() * 100
    
    plt.figure(figsize=(14, 5))
    plt.bar(annual_returns.index.year, annual_returns, color=[CSE_COLORS['positive'] if x > 0 else CSE_COLORS['negative'] for x in annual_returns])
    plt.title('ASPI Annual Returns (%)')
    plt.axhline(0, color='black', linewidth=1)
    
    plt.tight_layout()
    plt.show()
else:
    print("Market indices data not available.")

> **Insight - Indices**: The broader market (ASPI) shows distinct bull and bear cycles, heavily influenced by macroeconomic shocks (e.g., the 2022 crisis). The S&P SL 20 subset will generally track these movements but with potentially higher beta.

## Section 6: Time Series Analysis: Trading Volume
Assessing market liquidity trends.

In [ ]:
if not df_market_stats.empty and 'TURNOVER_EQUITY_Mn' in df_market_stats.columns:
    df_ms_modern = df_market_stats[(df_market_stats['Date'] >= '2001-01-01') & (df_market_stats['Date'] <= '2025-12-31')].set_index('Date')
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # Plot daily turnover
    ax.plot(df_ms_modern.index, df_ms_modern['TURNOVER_EQUITY_Mn'], color=CSE_COLORS['teal'], alpha=0.5, label='Daily Turnover')
    # Plot 90-day moving average of turnover
    turnover_ma90 = df_ms_modern['TURNOVER_EQUITY_Mn'].rolling(90).mean()
    ax.plot(df_ms_modern.index, turnover_ma90, color=CSE_COLORS['orange'], linewidth=2, label='90d MA Turnover')
    
    ax.set_title('Market Turnover (Equity) 2001-2025 in LKR Millions (2001-2025)')
    ax.set_ylabel('Turnover (Mn)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Market stats data not available.")

> **Insight - Volume**: Market liquidity surges dramatically during bull runs (e.g., 2010 post-war, 2021 pandemic recovery), providing excellent environments for momentum strategies.

## Section 7: Time Series Analysis: Individual Stocks
Analyzing top S&P SL 20 stocks over time using pre-calculated moving averages.

In [ ]:
all_stocks = liquidity_rank.index.tolist()

fig, axes = plt.subplots(5, 4, figsize=(24, 20))
axes = axes.flatten()

for i, stock in enumerate(all_stocks):
    stock_data = df[df['CompanyCode'] == stock].set_index('Date')
    if len(stock_data) > 0:
        axes[i].plot(stock_data.index, stock_data['Close'], color=CSE_COLORS['primary'], label='Close')
        # Use pre-calculated 90-day MA
        axes[i].plot(stock_data.index, stock_data['MA_90d'], color=CSE_COLORS['orange'], label='90d MA', linewidth=2)
        axes[i].set_title(f'{stock} Price History & Trend')
        axes[i].legend(loc='upper right')
        axes[i].set_ylabel('Price (Rs.)')

plt.tight_layout()
plt.show()

> **Insight - Stocks**: Blue-chip stocks demonstrate clear, prolonged trend phases. The pre-calculated 90-day Moving Average effectively filters out short-term noise, providing a solid baseline for trend detection.

## Section 8: Distribution Analysis
Understanding the statistical properties of S&P SL 20 stock returns.

In [ ]:
# Market-wide daily return distribution
df_returns = df.dropna(subset=['DailyReturn_Pct'])
returns = df_returns['DailyReturn_Pct'].values

plt.figure(figsize=(10, 6))
sns.histplot(returns, bins=100, kde=True, stat="density", color=CSE_COLORS['accent1'], 
             binrange=(-15, 15))

# Fit normal distribution
valid_returns = returns[np.isfinite(returns)]
visual_returns = valid_returns[(valid_returns >= -20) & (valid_returns <= 20)]

mu, std = stats.norm.fit(visual_returns)
x = np.linspace(-15, 15, 100)
p = stats.norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2, label=f'Normal Fit ($\mu={mu:.2f}\%, \sigma={std:.2f}\%)')
plt.xlim(-15, 15)

plt.title('Distribution of Daily Returns (%) vs Normal Distribution')
plt.xlabel('Daily Return (%)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

# Skewness and Kurtosis
print(f"Skewness: {stats.skew(valid_returns):.4f}")
print(f"Kurtosis: {stats.kurtosis(valid_returns):.4f}")

> **Insight - Distributions**: Even for highly liquid blue-chip stocks, returns exhibit significant excess kurtosis (fat tails). This implies extreme market moves (both up and down) happen more frequently than a normal bell curve would predict.

## Section 9: Correlation Analysis
Analyzing relationships between price, volume, and cross-stock correlations among the top 10 S&P SL 20 constituents.

In [ ]:
numeric_cols = ['Open', 'High', 'Low', 'Close', 'TradeVolume', 'ShareVolume', 'Turnover']

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Matrix of Price & Volume Features')
plt.tight_layout()
plt.show()

# Cross-stock correlation for top 10 stocks by volume
top_10 = df.groupby('CompanyCode')['ShareVolume'].sum().nlargest(10).index
pivot_returns = df[df['CompanyCode'].isin(top_10)].pivot_table(index='Date', columns='CompanyCode', values='DailyReturn_Pct', aggfunc='mean')

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_returns.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Cross-Stock Return Correlation (Top 10 S&P SL 20)')
plt.tight_layout()
plt.show()

> **Insight - Correlations**: While Open/High/Low/Close are perfectly correlated, volume metrics have very weak linear correlations with absolute price levels. Cross-stock correlations are broadly positive, highlighting systematic market risk (beta) driving the top companies together.

## Section 10: Outlier Analysis
Analyzing extreme market events across the S&P SL 20.

In [ ]:
Q1 = df_returns['DailyReturn_Pct'].quantile(0.25)
Q3 = df_returns['DailyReturn_Pct'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

outliers = df_returns[(df_returns['DailyReturn_Pct'] < lower_bound) | (df_returns['DailyReturn_Pct'] > upper_bound)]
print(f"Total Outliers detected (3x IQR): {len(outliers)} ({len(outliers)/len(df_returns)*100:.2f}%)")

plt.figure(figsize=(12, 5))
sns.boxplot(x=df_returns['DailyReturn_Pct'], color=CSE_COLORS['accent2'])
plt.axvline(lower_bound, color='red', linestyle='--')
plt.axvline(upper_bound, color='red', linestyle='--')
plt.title('Daily Returns Boxplot with Outlier Bounds')
plt.xlim(-20, 20)
plt.tight_layout()
plt.show()

# Outliers over time
outliers = outliers.copy()
outlier_counts = outliers.groupby('Year').size()

plt.figure(figsize=(12, 5))
outlier_counts.plot(kind='bar', color=CSE_COLORS['negative'])
plt.title('Number of Extreme Return Events per Year')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

> **Insight - Outliers**: Outlier frequency spikes during specific years (e.g., 2011, 2022) matching highly volatile market regimes.

## Section 11: Feature Relationship Analysis
Analyzing the relationship between trading activity and price movement magnitude.

In [ ]:
df_sample = df.sample(min(10000, len(df))).copy()
df_sample['LogVolume'] = np.log1p(df_sample['ShareVolume'])
df_sample['AbsReturn'] = np.abs(df_sample['DailyReturn_Pct'])

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_sample, x='LogVolume', y='AbsReturn', alpha=0.1, color=CSE_COLORS['primary'])
plt.title('Log Volume vs Absolute Daily Return (10k Sample)')
plt.xlabel('Log(Share Volume)')
plt.ylabel('Absolute Daily Return (%)')
plt.tight_layout()
plt.show()

> **Insight - Relationships**: There is a visible "volatility smile" effect where higher trading volumes are associated with larger absolute price movements.

## Section 12: Time-Based Feature Engineering
Analyzing calendar effects using the pre-calculated time features.

In [ ]:
dow_returns = df.groupby('DayOfWeek')['DailyReturn_Pct'].mean()
# Ensure correct order
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
dow_returns = dow_returns.reindex(days)

plt.figure(figsize=(8, 5))
sns.barplot(x=dow_returns.index, y=dow_returns.values, color=CSE_COLORS['teal'])
plt.title('Average Daily Return by Day of Week (%)')
plt.ylabel('Mean Return (%)')
plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.show()

monthly_returns = df.groupby('Month')['DailyReturn_Pct'].mean().reindex(range(1, 13), fill_value=0)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(10, 5))
sns.barplot(x=months, y=monthly_returns.values, color=CSE_COLORS['accent1'])
plt.title('Average Daily Return by Month (%)')
plt.ylabel('Mean Return (%)')
plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.show()

> **Insight - Calendar Effects**: Minor seasonality is visible (e.g., September/October historically weaker), but the effects are generally small and require rigorous statistical testing before being used as standalone signals.

## Section 13: Volatility Analysis
Utilizing the pre-calculated `Volatility_30d_Ann` feature to analyze risk across the S&P SL 20.

In [ ]:
# Mean annualized volatility per stock
mean_vol = df.groupby('CompanyCode')['Volatility_30d_Ann'].mean().sort_values(ascending=False)

print("Average 30-Day Annualized Volatility (%) by Stock:")
display(mean_vol)

plt.figure(figsize=(12, 6))
sns.barplot(x=mean_vol.index, y=mean_vol.values, palette=SECTOR_PALETTE[:20])
plt.title('Average Annualized Volatility by Company')
plt.ylabel('Volatility (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Plot volatility over time for JKH (a representative blue chip)
jkh_data = df[df['CompanyCode'] == 'JKH'].set_index('Date')
plt.figure(figsize=(14, 5))
plt.plot(jkh_data.index, jkh_data['Volatility_30d_Ann'], color=CSE_COLORS['negative'])
add_crash_bands(plt.gca())
plt.title('JKH 30-Day Rolling Annualized Volatility (2001-2025)')
plt.ylabel('Volatility (%)')
plt.tight_layout()
plt.show()

> **Insight - Volatility**: Volatility exhibits strong clustering. Even for the most stable company in Sri Lanka (JKH), volatility spikes massively during macroeconomic shocks (2001, 2008, 2022).

## Section 14: Trend Analysis
Analyzing long-term trends using moving averages.

In [ ]:
if not df_indices.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df_idx_modern.index, df_idx_modern['ASPI'], color='black', label='ASPI', alpha=0.5)
    
    mas = [50, 200]
    colors = [CSE_COLORS['teal'], CSE_COLORS['orange']]
    
    for ma, color in zip(mas, colors):
        df_idx_modern[f'MA_{ma}'] = df_idx_modern['ASPI'].rolling(ma).mean()
        ax.plot(df_idx_modern.index, df_idx_modern[f'MA_{ma}'], color=color, label=f'{ma}-day MA', linewidth=2)
        
    ax.set_title('ASPI Trend Analysis (50 & 200 Day MAs)')
    ax.legend()
    plt.tight_layout()
    plt.show()

> **Insight - Trends**: The 50/200 day moving average crossovers provide clear historical signals for major market regime shifts, identifying prolonged bull and bear markets.

## Section 15: Interactive Visualizations (Plotly)
*(Note: These will render in a live notebook environment)*

In [ ]:
if not df_indices.empty:
    plot_df = df_idx_modern.reset_index().dropna(subset=['ASPI'])
    
    fig = px.line(plot_df, x='Date', y='ASPI', title='Interactive ASPI Chart (2001-2025)')
    fig.update_xaxes(rangeslider_visible=True)
    fig.show()

> **Insight - Interactivity**: Plotly allows for deep dives into specific crash and rally dates.

## Section 16: Closing Summary

### Key Findings
1. **Dataset Integrity**: The extracted S&P SL 20 dataset spanning 2001-2025 is incredibly robust. It provides a dense matrix of ~89,000 trading days across the market's top blue chips, completely avoiding the sparsity and missing data issues of the 1990s and small-cap stocks.
2. **Distributions**: Stock returns show classic fat tails (excess kurtosis).
3. **Volatility Clustering**: Volatility regimes map exactly to Sri Lanka's macroeconomic events, even for the most stable companies.
4. **Pre-Calculated Features**: The dataset already contains validated technical features (MA_30d, MA_90d, Vol_30d), eliminating the need for complex rolling-window calculations in the modeling phase.

### Next Steps (Phase 2 & 3)
- Design the recommendation logic to classify stock states (e.g., Uptrend, Downtrend, Sideways) based on the relationship between `Close`, `MA_30d`, and `MA_90d`.
- Build the predictive/recommendation system notebook that operationalizes these insights into actionable trading signals for the S&P SL 20 universe.